In [1]:
%pip install requests pandas newspaper3k

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install serpapi

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import serpapi
import pandas as pd
from newspaper import Article
import time

# --- CONFIGURATION ---
API_KEY = "35ee400445b907086d64bf97daa4d3a981019d97a68f3a9f661bda1ffeb7daa5"
START_DATE = "2026-09-01"
END_DATE = "2026-10-03"

QUERIES = {
    # 1. Security & Strategic Affairs
    '"Indian Army" AND ("North East India" OR "Northeast" OR Assam OR Arunachal OR Manipur OR Meghalaya OR Mizoram OR Nagaland OR Sikkim OR Tripura) AND (security OR insurgency OR "counter-insurgency" OR AFSPA OR "Myanmar border")': "Security & Strategic Affairs",
    
    # 2. Regional News & Incident Reports
    '"Indian Army" AND ("North East India" OR "Northeast" OR Assam OR Arunachal OR Manipur OR Meghalaya OR Mizoram OR Nagaland OR Sikkim OR Tripura) AND (operations OR deployment OR patrol OR incident OR update)': "Regional News & Incident Reports",
    
    # 3. Development & Infrastructure
    '"Indian Army" AND ("North East India" OR "Northeast" OR Assam OR Arunachal OR Manipur OR Meghalaya OR Mizoram OR Nagaland OR Sikkim OR Tripura) AND (infrastructure OR development OR "Border Roads" OR BRO OR construction OR bridges)': "Development & Infrastructure",
    
    # 4. Society & Youth (Op Sadbhavana)
    '"Indian Army" AND ("North East India" OR "Northeast" OR Assam OR Arunachal OR Manipur OR Meghalaya OR Mizoram OR Nagaland OR Sikkim OR Tripura) AND ("Operation Sadbhavana" OR "civic action" OR "youth recruitment" OR community OR "medical camp")': "Society & Youth (Op Sadbhavana)",
    
    # 5. Sports & Achievements
    '"Indian Army" AND ("North East India" OR "Northeast" OR Assam OR Arunachal OR Manipur OR Meghalaya OR Mizoram OR Nagaland OR Sikkim OR Tripura) AND (sports OR tournament OR football OR athletes OR "Asian Games" OR championship)': "Sports & Achievements"
}

def scrape_article_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return None

def main():
    client = serpapi.Client(api_key=API_KEY)
    collected_data = []
    
    for query,category in QUERIES.items():
        print(f"\nFetching SerpApi results for: '{query}'...")
        
        formatted_query = f"{query} after:{START_DATE} before:{END_DATE}"
        
        try:
            results = client.search({
                "engine": "google_news",
                "q": formatted_query,
                "gl": "in",  
                "hl": "en"   
            })
            
            news_results = results.get("news_results", [])
            
        except Exception as e:
            print(f"SerpApi search failed for '{query}': {e}")
            continue
            
        for result in news_results:
            title = result.get("title")
            url = result.get("link")
            
            source_info = result.get("source", {})
            source = source_info.get("name") if isinstance(source_info, dict) else source_info
            
            date = result.get("date")
            
            if not url:
                continue
                
            print(f"  -> Scraping text for: {title[:40]}...")
            full_text = scrape_article_text(url)
            
            collected_data.append({
                "Title": title,
                "Original Text": full_text,
                "Source": source,
                "URL": url,
                "Date": date,
                "Category": category
            })
            
            time.sleep(1)

    # Convert in DataFrame
    df = pd.DataFrame(collected_data)
    
    if not df.empty:
        df = df.dropna(subset=["Original Text"])
        
        output_filename = "Northeast_Army_News_Phase1_SerpApi.csv"
        df.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"\nSuccessfully saved {len(df)} articles to {output_filename}")
    else:
        print("\nNo data was collected.")


main()


Fetching SerpApi results for: '"Indian Army" AND ("North East India" OR "Northeast" OR Assam OR Arunachal OR Manipur OR Meghalaya OR Mizoram OR Nagaland OR Sikkim OR Tripura) AND (security OR insurgency OR "counter-insurgency" OR AFSPA OR "Myanmar border")'...
  -> Scraping text for: Manipur: Two UKNA cadres killed, Two inj...
  -> Scraping text for: Manipur: 2 militants killed, 2 injured i...
  -> Scraping text for: Indian Army Chief Reviews Manipur Securi...
  -> Scraping text for: Indian Army chief General Dhiraj Seth re...
  -> Scraping text for: COAS General Dhiraj Seth reviews securit...
Failed to scrape https://www.aninews.in/news/national/general-news/coas-general-dhiraj-seth-reviews-security-situation-in-northeast-during-2-day-visit20260729185941: Article `download()` failed with 403 Client Error: Forbidden for url: https://www.aninews.in/news/national/general-news/coas-general-dhiraj-seth-reviews-security-situation-in-northeast-during-2-day-visit20260729185941 on URL https:/

In [4]:
import pandas as pd

TARGET_PER_CATEGORY = 9

def get_random_essential_articles(input_filename, output_filename):
    print(f"Loading data from {input_filename}...\n")
    
    try:
        df = pd.read_csv(input_filename)
    except FileNotFoundError:
        print(f"Error: Could not find {input_filename}")
        return
    
    df = df.drop_duplicates(subset=['URL'], keep='first')

    final_articles = []
    categories = df['Category'].dropna().unique()
    
    for category in categories:
        category_df = df[df['Category'] == category]
        actual_count = len(category_df)
        
        if actual_count >= TARGET_PER_CATEGORY:
            sliced_df = category_df.sample(n=TARGET_PER_CATEGORY)
            final_articles.append(sliced_df)
            print(f"[{category}] Success: Randomly selected 9 articles.")
        else:
            final_articles.append(category_df)
            print(f"[{category}] WARNING: Only found {actual_count} articles. Kept all of them.")

    if final_articles:
        final_df = pd.concat(final_articles)

        #Details in csv
        final_df.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"\nSuccessfully saved {len(final_df)} total randomized articles to {output_filename}")
    else:
        print("No articles to save. Check your input CSV.")

if __name__ == "__main__":
    INPUT_FILE = "Northeast_Army_News_Phase1_SerpApi.csv" 
    OUTPUT_FILE = "Magazine_9_Per_Category_Random.csv"
    
    get_random_essential_articles(INPUT_FILE, OUTPUT_FILE)

Loading data from Northeast_Army_News_Phase1_SerpApi.csv...

[Security & Strategic Affairs] Success: Randomly selected 9 articles.
[Regional News & Incident Reports] Success: Randomly selected 9 articles.
[Development & Infrastructure] Success: Randomly selected 9 articles.
[Society & Youth (Op Sadbhavana)] Success: Randomly selected 9 articles.
[Sports & Achievements] Success: Randomly selected 9 articles.

Successfully saved 45 total randomized articles to Magazine_9_Per_Category_Random.csv
